In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Reading

In [0]:
df = spark.read.format("parquet")\
                .load("abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/customers")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")
display(df)

In [0]:
df = df.withColumn("domains", split(col("email"),"@")[1])

display(df)

In [0]:
df1 = df.groupBy("domains").agg(count("customer_id").alias("total_customers")).orderBy(col("total_customers").desc())

# df = df.groupBy("domains").agg(count("customer_id").alias("total_customers").sort("total_customers", ascending=False)) # Will do same as orderBy

display(df1)

In [0]:
import time

df_gmail = df.filter(col("domains").contains("gmail"))
# df_gmail = df.filter(col("domains") == "gmail.com")

display(df_gmail)

time.sleep(5)

df_yahoo = df.filter(col("domains") == "yahoo.com")

display(df_yahoo)

time.sleep(5)

df_hotmail = df.filter(col("domains") == "hotmail.com")
display(df_hotmail)

time.sleep(5)

In [0]:
df = df.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
df = df.drop('first_name', 'last_name')

display(df)

In [0]:
df.write.format("delta")\
        .mode("append")\
        .save("abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/customers")